# A tour of Foil-Lab

Runs on a CPU in under a minute. It covers sections and flaps, inviscid and viscous polars, a
comparison with a committed XFOIL reference polar, the corrections, and tables. Install with
`pip install "foil-lab[pandas]"` (or `pip install -e ".[all]"` in a clone).

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import foil_lab as fl
from foil_lab import plot

t0 = time.perf_counter()
af = fl.Airfoil.naca("2412")
flapped = af.with_flap(hinge_x=0.75, deflection=15.0)
fig, ax = plot.geometry(fl.Configuration((fl.Element(af), fl.Element(flapped, offset=(0.0, -0.25)))))
ax.set_title("NACA 2412, clean and with a 25 % plain flap at 15 deg");

## Inviscid analysis: surface pressure and the effect of a flap

In [ ]:
res = fl.analyze(af, alpha=4.0)
print(f"Cl {res.cl:.4f}  Cm {res.cm:.4f}  critical Mach {res.mach_critical:.3f}")
for d in (0.0, 5.0, 10.0, 15.0):
    r = fl.analyze(af.with_flap(0.75, d), alpha=4.0)
    print(f"flap {d:4.1f} deg: Cl {r.cl:.3f}  hinge moment {r.ch:+.4f}")
fig, ax = plot.cp([res, fl.analyze(flapped, 4.0)], labels=["clean", "flap 15 deg"]);

## Viscous polar against XFOIL

The coupled viscous-inviscid solver at Re = 1e6 (the section repanelled to 160 panels by arc length,
the default), next to the committed XFOIL 6.99 reference polar of the same section. The full
comparison, on identical coordinate files for both codes, is in `VALIDATION.md`.

In [ ]:
from foil_lab.benchmark.references import load_reference

pol = fl.polar(af, [0.0, 2.0, 4.0, 6.0], re=1e6)
ref = load_reference("NACA2412", 1e6, 160)
for r in pol.results:
    x = ref.rows[r.alpha]
    print(f"alpha {r.alpha:3.0f}: Cl {r.cl:.4f} (XFOIL {x.cl:.4f})  Cd {r.cd:.5f} (XFOIL {x.cd:.5f})  "
          f"converged {r.converged}")
fig, axes = plot.polars(pol, labels=["Foil-Lab, Re 1e6"], reference=ref)

## Corrections

Applied once each, in a fixed order, and listed in the result: ground effect, sweep,
compressibility (Kármán-Tsien, inviscid), finite aspect ratio.

In [ ]:
r = fl.analyze(af, alpha=4.0, mach=0.5, sweep=20.0, aspect_ratio=8.0)
print(f"CL {r.cl:.4f}  CDi {r.cd_induced:.5f}  critical Mach {r.mach_critical:.3f}")
for c in r.corrections:
    print(" -", c)
g = fl.analyze(af, alpha=4.0, ground_height=0.3)
print(f"in ground effect at h/c = 0.3: Cl {g.cl:.4f} (free air {res.cl:.4f})")

## Tables

In [ ]:
df = pol.to_dataframe()
print(df[["alpha", "cl", "cd", "cm", "xtr_upper", "converged"]].to_string(index=False))
print(f"notebook run time {time.perf_counter() - t0:.0f} s")